# Tutorial 9 — experimental self-scaled optimizers

Compare **Reference L-BFGS**, **SSBFGS** and **SSBroyden** on the first tutorial's one-dimensional traveling wave. This is an additional controlled experiment, with identical actual starting weights, normalized data, scalar objective and FP64 precision. The original eight tutorials and supported NIF optimizer API remain available unchanged.

The default **FULL** run performs a fresh 5,000-epoch shared AdaBelief warmup, followed by one round of at most 1,000 quasi-Newton iterations for each method. This differs from original T1's FP32, 200-round L-BFGS workload. Budget roughly half an hour or more on a laptop-class CPU; Colab speeds vary. A single dense inverse Hessian requires **261.4 MiB**, with larger temporary memory use. A standard CPU runtime is sufficient; this comparison does not use the GPU.

Set the environment variable `NIF_SMOKE=1` before execution only for an explicitly labeled two-epoch/three-iteration development check. SMOKE results are not scientific convergence evidence.


In [ ]:
import os

SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
MODE = "SMOKE" if SMOKE else "FULL"
print("Selected workload:", MODE)
print("FULL: 5,000 shared warmup epochs, then at most 1,000 QN iterations per method.")
print("SMOKE: 2 shared warmup epochs, then at most 3 QN iterations per method.")


## 1. Prepare a pinned, separate environment

This cell uses Python 3.11–3.13. It installs NIF **v2.0.0** and the comparison dependencies into a separate virtual environment, then runs the experiment in that environment. NumPy or PyTorch already imported by the notebook kernel is left intact. Existing matching system packages can be reused, while pinned overrides live inside the virtual environment. Colab does not require `ensurepip`: the parent pip (22.3+) manages only the explicitly selected child interpreter.

The support code is obtained from the immutable NIF commit printed below. The optimizer authors' source is separately pinned and SHA256-verified at runtime, with no replacement of installed SciPy files.


In [ ]:
import hashlib
import json
import subprocess
import sys
import tempfile
import venv
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use a Python 3.11–3.13 runtime for this pinned experiment.")

SUPPORT_REF = "40a0a212d5d2d38078a21d0cb27440439e8c5ae4"  # Immutable experiment implementation.
NIF_CORE_REF = "v2.0.0"
REPOSITORY = "https://github.com/pswpswpsw/nif.git"
local_override = os.environ.get("NIF_SS_LOCAL_REPO")  # Optional development checkout, not needed in Colab.
if local_override:
    support_root = Path(local_override).resolve()
else:
    support_root = Path(tempfile.gettempdir()) / ("nif-ss-support-" + SUPPORT_REF[:12])
    if not support_root.exists():
        subprocess.check_call(["git", "init", "--quiet", str(support_root)])
        subprocess.check_call(["git", "-C", str(support_root), "remote", "add", "origin", REPOSITORY])
        subprocess.check_call(["git", "-C", str(support_root), "fetch", "--quiet", "--depth", "1", "origin", SUPPORT_REF])
        subprocess.check_call(["git", "-C", str(support_root), "checkout", "--quiet", "--detach", "FETCH_HEAD"])

benchmark = support_root / "examples" / "self_scaled" / "benchmark.py"
requirements = benchmark.with_name("requirements.txt")
if not benchmark.is_file() or not requirements.is_file():
    raise RuntimeError("The selected checkout does not contain the self-scaled tutorial support files.")
support_commit = subprocess.check_output(["git", "-C", str(support_root), "rev-parse", "HEAD"], text=True).strip()
if not local_override and SUPPORT_REF != "main" and support_commit != SUPPORT_REF:
    raise RuntimeError("The support checkout does not match the pinned commit.")
if not local_override:
    subprocess.check_call(["git", "-C", str(support_root), "diff", "--exit-code", "HEAD", "--", "examples/self_scaled"])

requirements_hash = hashlib.sha256(requirements.read_bytes()).hexdigest()
environment = Path(tempfile.gettempdir()) / ("nif-ss-env-v2-" + str(sys.version_info[0]) + str(sys.version_info[1]) + "-" + requirements_hash[:12])
SS_PYTHON = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not SS_PYTHON.is_file():
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
prefixes = json.loads(subprocess.check_output([str(SS_PYTHON), "-c", "import json,sys; print(json.dumps([sys.prefix,sys.base_prefix]))"], text=True))
if Path(prefixes[0]).resolve() != environment.resolve() or prefixes[0] == prefixes[1]:
    raise RuntimeError("The child interpreter is not a ready isolated virtual environment.")

pip = [sys.executable, "-m", "pip", "--python", str(SS_PYTHON), "install", "--quiet", "--disable-pip-version-check"]
subprocess.check_call([*pip, "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.14.1"])
subprocess.check_call([*pip, "-r", str(requirements)])
subprocess.check_call([*pip, "--no-deps", "--ignore-installed", "git+" + REPOSITORY + "@" + NIF_CORE_REF])

child_environment = dict(os.environ)
child_environment.pop("PYTHONPATH", None)
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    child_environment[name] = "2"
versions = subprocess.check_output(
    [str(SS_PYTHON), "-c", "import json,sys,nif,torch,numpy,scipy,matplotlib; print(json.dumps(dict(python=sys.version.split()[0],nif=nif.__version__,torch=torch.__version__,numpy=numpy.__version__,scipy=scipy.__version__,matplotlib=matplotlib.__version__)))"],
    env=child_environment, text=True, cwd=tempfile.gettempdir(),
)
print("Support commit:", support_commit)
print("NIF core ref:", NIF_CORE_REF)
print("Requirements SHA256:", requirements_hash)
print("Isolated environment versions:", versions.strip())


## 2. Shared scientific case and controlled training

The field is

$$
z=x-0.2-0.006t,\qquad u(t,x)=e^{-1000z^2}\sin(4z).
$$

All 2,000 original training rows are retained, with columns `[t,x]`. The stored normalized arrays and original statistics are uniformly cast to FP64. A common 30-unit/two-hidden-layer Swish NIF is warmed up once; each arm then reloads the **same actual checkpoint**, not merely a shared seed. The first objective and every component of its first gradient are hashed and compared.

Each quasi-Newton evaluation uses the fixed full-data scalar objective
`mean((prediction-normalized_target)**2) + model.regularization_loss(features)`.
L-BFGS uses Hager–Zhang line search and 20 correction pairs. The pinned author SSBFGS/SSBroyden code uses dense inverse-Hessian updates and Strong Wolfe line search. These are full-batch CPU experiments with different stopping contracts.

The next cell performs the real computation. It creates a new output directory and does not display cached historical results. Runtime progress appears every 100 warmup epochs or quasi-Newton iterations.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_DIR = (Path.cwd() / "pytorch_outputs" / "self-scaled" / (MODE.lower() + "-" + run_name)).resolve()
command = [str(SS_PYTHON), str(benchmark), "--output", str(RUN_DIR)]
if SMOKE:
    command.append("--smoke")
subprocess.check_call(command, cwd=tempfile.gettempdir(), env=child_environment)
print("New result directory:", RUN_DIR)


## 3. Physical absolute errors, timing and raw termination status

Training-grid errors compare to the original stored physical field at 10×200 points. The **additional** 45×800 interpolation grid compares to the analytical field within `t∈[0,90],x∈[0,.995]`; it is a different evaluation set. Casting the source FP32 arrays to FP64 does not recover their original rounding error.

QN times exclude the shared warmup, prediction and plotting. They include line search, full-data objective/gradient calls, curvature updates and logging. A single CPU run compares complete implementations; it does not establish a universal speed ranking. An iteration limit, precision loss or invalid curvature state is displayed as such, rather than called convergence.


In [ ]:
from IPython.display import Markdown, display

summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
assert summary["mode"] == MODE
assert summary["shared_start_verified"], "Actual starting weights/loss/gradient differed."
print("Actual shared starting parameter hash:", manifest["shared_start_parameter_hash"])
print("Shared AdaBelief warmup seconds:", summary["warmup_elapsed_seconds"])
print("Recorded CPU threads:", manifest["threads"])

report_text = (RUN_DIR / "REPORT.md").read_text(encoding="utf-8")
display(Markdown(report_text.split("## Configuration and provenance")[0]))
for method, arm in summary["arms"].items():
    print(method, "raw termination:", json.dumps(arm["status"], indent=2))


## 4. Original-style field plots and convergence

These figures are redrawn from **this run's saved predictions**. Each original-style figure uses the original tutorial's `contourf`, 50 levels, `seismic`, fixed normalized limits `[-5,5]`, and signed residual `prediction-truth`. Time is horizontal and position vertical. Before any reshape, the support code checks the full time-major/x-fast coordinate pairing point by point.

Failed arms without a returned finite field remain in the status table and are omitted from field plots. The convergence plot shows accepted states; it does not interpolate or extrapolate an earlier-ending method.


In [ ]:
from IPython.display import Image

figures = json.loads((RUN_DIR / "report_artifacts.json").read_text(encoding="utf-8"))
for name in ("lbfgs_original_style", "ssbfgs_original_style", "ssbroyden_original_style", "convergence"):
    if name in figures:
        display(Image(filename=str(RUN_DIR / figures[name])))


## 5. Additional physical-field and absolute-error comparisons

The next plots use shared color scales across methods. Physical values and absolute errors are labeled separately from the normalized original-style presentation. The 45×800 grid contains additional model predictions, including new times between the training times; it is not display interpolation of the original 10×200 values.


In [ ]:
for name in ("train_physical_comparison", "train_absolute_errors", "dense_physical_comparison", "dense_absolute_errors"):
    if name in figures:
        display(Image(filename=str(RUN_DIR / figures[name])))


## 6. Keep the evidence and interpret the result

The output directory contains actual shared/final checkpoints, initial/final vectors, data and predictions, objective-evaluation histories, accepted-iteration histories, versions, source hashes, and `REPORT.md`. The next cell saves a small source/environment receipt alongside them.

Judge training error and interpolation error separately. A lower sampled loss need not give a better continuous field. Compare final physical errors, actual termination status, wall time and objective-evaluation count together. Repeated seeds and matched wall-time budgets would be needed for broader optimizer conclusions.

The author source is pinned to [commit 990a82c](https://github.com/EliKiani/Optimizing_the_Optimizer_PINNs/tree/990a82c1353cbda9f5afee3200b222fdb821c521), accompanying [arXiv:2501.16371](https://arxiv.org/abs/2501.16371). It is downloaded into a separate verified cache; no SciPy source is overwritten. See [the experiment guide](https://github.com/pswpswpsw/nif/tree/main/examples/self_scaled) for the exact scientific and source-provenance contracts.


In [ ]:
receipt = dict(mode=MODE, support_commit=support_commit, core_ref=NIF_CORE_REF,
               requirements_sha256=requirements_hash, environment=json.loads(versions),
               author_source=manifest["author"], shared_start_verified=summary["shared_start_verified"])
(RUN_DIR / "notebook_receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print(json.dumps(receipt, indent=2))
print("Saved REPORT.md, manifest.json, summary.json, predictions, checkpoints and figures in:", RUN_DIR)
